# 01 — AFCP-EM Expert Matching Baseline (AFCP-EM-Expert track)

> **🚧 Skeleton — Use Case 2 (planned for 2026-1).**
> Data loads below are real; the compliance-gate, matching, and metric blocks
> raise `NotImplementedError` until implemented. Mirrors the structure of
> [`04_prior_art_baseline.ipynb`](04_prior_art_baseline.ipynb).

Deliverable ② + dissertation seed (Park 2026). Loads the 110-curated expert
pool, the 50 technology-problem set, the 3-rater 7,800-rating GT, and the
KR + US governance masters, then will report **MRR, NDCG@5, leakage_rate@5**
under multi-jurisdiction compliance gating.

**Run order:**

```bash
make experts          # 100 synthetic expert profiles
make curated-experts  # 110 curated profiles (Park 2026a)
make curated-ratings  # 7,800 3-rater ratings + κ/ICC
make compliance       # KR + US governance instances (205 triples)
make sirp-problems    # 50 problems + 25 regulatory scenarios
jupyter nbconvert --to notebook --execute notebooks/01_matching_baseline_afcp.ipynb
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'data' / 'experts' / 'curated_profiles.parquet').exists():
    pass
elif (ROOT.parent / 'data' / 'experts' / 'curated_profiles.parquet').exists():
    ROOT = ROOT.parent
else:
    raise SystemExit('Run `make expdataset` first to populate data/experts/ and data/compliance/.')

experts = pd.read_parquet(ROOT / 'data' / 'experts' / 'curated_profiles.parquet')
problems = pd.read_parquet(ROOT / 'data' / 'problems.parquet')
ratings = pd.read_parquet(ROOT / 'data' / 'experts' / 'curated_ratings.parquet')
kr_controls = json.loads((ROOT / 'data' / 'compliance' / 'kr_standards_v1.json').read_text())
us_controls = json.loads((ROOT / 'data' / 'compliance' / 'us_standards_v1.json').read_text())
leakage = json.loads((ROOT / 'data' / 'compliance' / 'leakage_incidents_v1.json').read_text())

print('experts:', experts.shape, ' problems:', problems.shape, ' ratings:', ratings.shape)
print('KR controls:', len(kr_controls.get('controls', kr_controls)),
      ' US controls:', len(us_controls.get('controls', us_controls)),
      ' leakage incidents:', len(leakage.get('incidents', leakage)))


## TODO — Compliance gate (architectural, not post-hoc)

For each `(expert, problem)` pair, filter out matches that would create
a compliance leak under the active governance regime **before** scoring:

- **KR ITPA §33 / §34** — national-core-tech (NCT) designation: matching is blocked when the expert's nationality / affiliation crosses the NCT boundary for the problem's technology.
- **US EAR / CCL** — Deemed Export: matching is blocked when the expert's nationality is on a restricted country list for the technology's ECCN.
- **Leakage incidents L1–L4** (`data/compliance/leakage_incidents_v1.json`) — treat each historical case as an adversarial test that the gate must catch.

See [`docs/leakage_protocol.md`](../docs/leakage_protocol.md) for the formal definition.


In [ ]:
def compliance_gate(expert: pd.Series,
                     problem: pd.Series,
                     kr_controls: dict,
                     us_controls: dict) -> bool:
    """Return True if (expert, problem) is allowed; False if it leaks.

    Architectural, not post-hoc: this runs before the matching score and
    masks the candidate pool, so leakage_rate@K is bounded by gate recall,
    not by a downstream filter.
    """
    raise NotImplementedError(
        'KR ITPA + US EAR jurisdiction-crossing check. '
        'See docs/leakage_protocol.md.'
    )


## TODO — Matching score over the gated pool

TF-IDF on `expertise_summary` (expert) × `problem_description` (problem)
as the floor baseline; replace with sentence-transformer embeddings later.
Only score `(expert, problem)` pairs for which `compliance_gate` returned True.


In [ ]:
def score_pairs(experts: pd.DataFrame,
                 problems: pd.DataFrame,
                 kr_controls: dict,
                 us_controls: dict) -> pd.DataFrame:
    """Return DataFrame[expert_id, problem_id, score] over the gated pool."""
    raise NotImplementedError(
        'TF-IDF on expertise_summary x problem_description; '
        'mask out pairs that fail compliance_gate before scoring.'
    )


## TODO — Metrics: MRR / NDCG@5 / leakage_rate@5

Ground truth: `curated_ratings.parquet` (7,800 ratings, 3-rater consensus,
Fleiss κ = 0.258, ICC(2,1) = 0.552). A rating ≥ 4 on the 1–5 scale is treated
as positive.

- **MRR / NDCG@5 / Recall@K** — retrieval quality
- **leakage_rate@5** — fraction of top-5 results that fail the (independently re-checked) compliance gate. *Architectural target: 0.0.*
- **Stratification** — report metrics by (jurisdiction, ECCN bucket, NCT-designated?) so we can see where the gate trades off recall.


In [ ]:
raise NotImplementedError(
    'Compute MRR, NDCG@5, Recall@K, leakage_rate@5 against the '
    '7,800 3-rater GT. Mirror the metric shape from '
    'notebooks/04_prior_art_baseline.ipynb cell 4.'
)


## Notes

- This notebook is the **first application of the lab agenda**: AFCP-EM Expert matching for SDKB Use Case 2. It is the dissertation-seed implementation (Park 2026).
- **Architectural compliance**: the gate runs *before* scoring. This is the distinguishing claim of AFCP-EM vs post-hoc filtering approaches.
- The 7,800 3-rater GT (κ = 0.258, ICC = 0.552) is *moderate* inter-rater agreement — report metrics with CIs derived from the 3-rater spread; do not over-claim.
- See also [`05_synthetic_vs_curated_comparison.ipynb`](05_synthetic_vs_curated_comparison.ipynb) for the GT-validity diagnostics that should accompany every metric reported here.
- For the full sweep across all 110 experts × 50 problems, mirror this logic into `scripts/evaluate_expert_matching.py`.
